# 19 Degrees: reproduce the core results

This notebook reproduces the two headline findings of the 19 Degrees page from its downloadable data, in a few minutes:

1. **The 19°C line:** no bull shark bite in Australia since 1900 happened in a month when the open sea was below 19°C.
2. **The move into coastal waters:** bull shark bites have moved out of estuaries, harbors and rivers into coastal waters.

Download `bite-temperatures.csv` and `bull-bites-open-coast.csv` from the page (Researchers → Download the data) into the same folder as this notebook. You need Python 3 with pandas, scipy and matplotlib.

Data: Australian Shark-Incident Database (Taronga Conservation Society Australia and Flinders University) and HadISST sea surface temperature (UK Met Office). Please cite both, and the page, if you reuse them.

In [ ]:
import pandas as pd
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

bites = pd.read_csv('bite-temperatures.csv')   # one row per bite: species, sea_temp_c
bites.head()

## 1. Bites below 19°C, by species

`sea_temp_c` is that month's average sea surface temperature for the 1° square of open sea nearest the bite (about 110 km across). It describes the season, not the exact water the person was in.

In [ ]:
summary = (bites.assign(below_19=bites.sea_temp_c < 19)
           .groupby('species').below_19.agg(bites='size', below_19='sum'))
summary['share_below_%'] = (summary.below_19 / summary.bites * 100).round(1)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for sp, colour in [('bull', '#F0A04B'), ('tiger', '#F5C46B'), ('white', '#5AB4E5')]:
    ax.hist(bites.loc[bites.species == sp, 'sea_temp_c'], bins=range(12, 31), alpha=.6, label=sp, color=colour)
ax.axvline(19, color='k', ls='--', lw=1)
ax.set_xlabel('Sea temperature that month (°C)'); ax.set_ylabel('Bites'); ax.legend(); plt.show()

**Try next:** how sensitive is the rule to the threshold? Count bull shark bites below 18.5, 19.5 and 20°C. What is the coolest month any bull shark bite happened in?

In [ ]:
bites.loc[bites.species == 'bull', 'sea_temp_c'].describe()

## 2. The move into coastal waters

Each row is one era. `open_coast_bites` are bites in coastal waters (the shark-incident database's "coastal" sites, mostly surf beaches), not in an estuary, harbor, river or lake. `set` is `all` bites or only those where the species was `confirmed` by physical evidence.

In [ ]:
oc = pd.read_csv('bull-bites-open-coast.csv')
aus = oc[(oc.area == 'Australia') & (oc.set == 'all')].copy()
aus['share_open_%'] = (aus.open_coast_bites / aus.bites * 100).round(0)
aus

In [ ]:
first, last = aus.iloc[0], aus.iloc[-1]
table = [[last.open_coast_bites, last.bites - last.open_coast_bites],
         [first.open_coast_bites, first.bites - first.open_coast_bites]]
odds, p = fisher_exact(table, alternative='greater')
print(f'{int(first.era_start)}-{int(first.era_end)}: {first.open_coast_bites}/{first.bites} coastal')
print(f'{int(last.era_start)}-{int(last.era_end)}: {last.open_coast_bites}/{last.bites} coastal')
print(f'One-sided Fisher exact test: p = {p:.2g}')

## Your turn

Pick one of the page's twenty-two testable claims (Researchers → Claims), state in advance what result would support it and what would rule it out, and test it. Send your claim ID, data source, method, result and code through the page's contact form (topic: Research or collaboration). Accepted results are credited by name on the claim and in the page's changelog.